# flag column analysis

## Load data

In [ ]:
import re
import pandas as pd

LOG_PATH = "HPC.log"
COLUMNS = ["event_id", "node", "component", "state", "time", "flag", "message"]

LINE_RE = re.compile(r"^(\d+) (.*?) (\S+) (\S+) (\d+(?:\.\d+)?) (-?\d)(?: (.*))?$")


def load_hpc_log(path=LOG_PATH):
    with open(path, "rb") as f:
        lines = [l for l in f.read().decode("ascii").split("\r\n") if l]

    rows, unparsed = [], []
    for line in lines:
        m = LINE_RE.match(line)
        if m:
            rows.append(m.groups())
        else:
            unparsed.append(line)
    
    df = pd.DataFrame(rows, columns=COLUMNS)
    df["message"] = df["message"].fillna("")
    df["event_id"] = df["event_id"].astype("int64")
    df["flag"] = df["flag"].astype("int8")
    t = pd.to_numeric(df["time"], errors="coerce")
    df["bad_time"] = ~df["time"].str.fullmatch(r"\d{10}")
    df["ts"] = pd.to_datetime(t.where(~df["bad_time"]), unit="s")
    df["time"] = t
    df["row"] = range(len(df))
    return df, unparsed

In [ ]:
df, unparsed = load_hpc_log()
print(f"{len(df):,} rows, {len(unparsed)} unparsed, {df.bad_time.sum()} bad timestamps")
df.head()

## flag values

In [ ]:
df.flag.value_counts()

## flag by component and state

In [ ]:
t = df.groupby(["component", "state", "flag"]).size().unstack("flag", fill_value=0)
t[(t[0] > 0) | (t[-1] > 0)].sort_values(by=[-1, 0, 1], ascending=False)

## flag by message
All messages (numbers replaced by placeholders) that ever appear with flag 0 or -1.

In [ ]:
msg = df.message.str.replace(r"0x[0-9a-fA-F]+", "<HEX>", regex=True).str.replace(r"\d+", "<N>", regex=True)
m = df.groupby(["component", "state", msg, "flag"]).size().unstack("flag", fill_value=0)
print(len(m), "message patterns in total,", ((m[0] > 0) | (m[-1] > 0)).sum(), "with flag 0 or -1")
m[(m[0] > 0) | (m[-1] > 0)].sort_values(by=[-1, 0, 1], ascending=False)

## Identical copies of the same row, by flag
`copies` = number of rows with the same node, component, state, time and message.

In [ ]:
key = ["node", "component", "state", "time", "message"]
df["copies"] = df.groupby(key).flag.transform("size")
df.groupby("flag").copies.describe()

In [ ]:
df.groupby("flag").copies.apply(lambda x: (x == 1).mean()).rename("share of rows with no duplicate")

## Which flags appear together among identical copies

In [ ]:
counts = pd.get_dummies(df.flag).rename(columns={1: "flag_1", 0: "flag_0", -1: "flag_neg1"})
groups = pd.concat([df[key], counts], axis=1).groupby(key).sum()
groups[["flag_1", "flag_0", "flag_neg1"]].gt(0).value_counts()

## Groups that mix flag 0 and flag 1: how many flag 1 copies do they have

In [ ]:
groups[(groups.flag_0 > 0) & (groups.flag_1 > 0)].flag_1.value_counts()

## Example: one node around the moment with the most flag 0 rows

In [ ]:
top_time = df[df.flag == 0].groupby("time").size().idxmax()
example_node = df[(df.time == top_time) & (df.flag == 0)].node.iloc[0]
window = df[(df.node == example_node) & df.time.between(top_time - 86400, top_time + 86400)]
window.sort_values("time")[["event_id", "node", "component", "state", "ts", "flag", "message"]]

## Example: one node around the moment with the most flag -1 rows

In [ ]:
top_time = df[df.flag == -1].groupby("time").size().idxmax()
example_node = df[(df.time == top_time) & (df.flag == -1)].node.iloc[0]
window = df[(df.node == example_node) & df.time.between(top_time - 86400, top_time + 86400)]
window.sort_values("time")[["event_id", "node", "component", "state", "ts", "flag", "message"]]

## flag over time

In [ ]:
pd.crosstab(df.ts.dt.to_period("Q"), df.flag)

## Conclusions
- `flag` takes the values 1 (418,546 rows), 0 (11,779) and -1 (3,165). It is undocumented, so these are observations, not a confirmed meaning.
- Rows with flag 0 or -1 are almost always repeated copies of the same row (same node, time, message and `event_id`). Only 8.7% of flag 0 rows and 0.7% of flag -1 rows have no duplicate, against 94.5% of flag 1 rows. The median number of copies is 3 for flag 0, 4 for flag -1 and 1 for flag 1.
- When identical copies mix flag 0 and flag 1 (3,287 groups), exactly one copy has flag 1 and the rest have flag 0.
- Flag -1 never shares a group with flag 1.
- `flag` does not track severity. Flags 0 and -1 appear on 15 of 403 message patterns, almost all of them routine state reports (`running`, `configured out`, `not responding`, `blocked`, `active`), and each of those messages also appears with flag 1.
- After collapsing identical copies, 6,232 of 411,483 distinct events carry flag 0 or -1.
- Decision: do not use `flag` as an error label. Drop identical copies before any analysis, which removes most of the flag 0 and -1 rows.